In [1]:
# Enable IPython autoreload for iterative development
%load_ext autoreload
%autoreload 2

# Download All Annotated Article PDFs

Download article PDFs for all records in `dataset_092624_validated.xlsx` (Dryad + Zenodo + Semantic Scholar) to `data/pdfs/fuster/` using the full fallback chain:

1. **Strategy 1**: Pre-fetched `pdf_url` from xlsx (WU-A3 / OpenAlex)
2. **Strategy 2**: Unpaywall API
3. **Strategy 3**: University EZproxy authentication
4. **Strategy 4**: Sci-Hub

Downloads ALL records regardless of OA status. Ends with a synthesis table segmented by source.

In [2]:
# Section 1: Imports and configuration
from pathlib import Path
import pandas as pd
import logging
import time

from llm_metadata.pdf_download import download_pdf_with_fallback
from llm_metadata.ezproxy import extract_cookies_from_browser

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("download_all_fuster_pdfs")

# Output directory for all PDFs
OUTPUT_DIR = Path("data/pdfs/fuster")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Manifest path:    {MANIFEST_PATH}")

Output directory: C:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster
Manifest path:    data\pdfs\fuster\manifest.csv


## Section 2: Load validated dataset (all sources)

Load `dataset_092624_validated.xlsx` (output of WU-A3) and filter to records with a `cited_article_doi`.
The `pdf_url` and `is_oa` columns are already populated from OpenAlex — no separate API fetch needed.

In [3]:
# Load validated xlsx (all sources: Dryad, Zenodo, Semantic Scholar)
xlsx_path = Path("data/dataset_092624_validated.xlsx")
assert xlsx_path.exists(), f"Validated xlsx not found: {xlsx_path}"

df = pd.read_excel(xlsx_path, dtype=str)
df.columns = [c.strip() for c in df.columns]

# Normalize string columns
for col in ['cited_article_doi', 'pdf_url', 'source', 'id', 'title']:
    df[col] = df[col].fillna('').str.strip()

# Convert is_oa to bool (stored as string in xlsx)
df['is_oa'] = df['is_oa'].map({'True': True, 'False': False})  # NaN → NaN for unknown

# Filter to records with a cited article DOI
annotated_df = df[df['cited_article_doi'] != ''].copy()
annotated_df.reset_index(drop=True, inplace=True)

print(f"Total records in xlsx: {len(df)}")
print(f"Records with cited_article_doi: {len(annotated_df)}")
print(f"\nBreakdown by source:")
print(annotated_df['source'].value_counts())
print(f"\npdf_url already populated (WU-A3): {(annotated_df['pdf_url'] != '').sum()}")
print(f"pdf_url missing (full chain needed): {(annotated_df['pdf_url'] == '').sum()}")

annotated_df[['id', 'cited_article_doi', 'source', 'is_oa', 'pdf_url']].head(5)

Total records in xlsx: 299
Records with cited_article_doi: 250

Breakdown by source:
source
semantic_scholar    175
zenodo               38
dryad                37
Name: count, dtype: int64

pdf_url already populated (WU-A3): 103
pdf_url missing (full chain needed): 147


,id,cited_article_doi,source,is_oa,pdf_url
0,4,https://doi.org/10.5281/zenodo.5898699,zenodo,True,
1,5,https://doi.org/10.1093/jhered/esx103,zenodo,True,https://academic.oup.com/jhered/article-pdf/10...
2,9,https://doi.org/10.1371/journal.pone.0128238,dryad,True,https://journals.plos.org/plosone/article/file...
3,11,https://doi.org/10.1371/journal.pone.0073695,zenodo,True,https://journals.plos.org/plosone/article/file...
4,12,https://doi.org/10.1002/ece3.4685,zenodo,True,https://onlinelibrary.wiley.com/doi/pdfdirect/...


## Section 3: Download PDFs (all records, all OA statuses)

For each record with a `cited_article_doi`:
- Pass `pdf_url` from xlsx as Strategy 1 (pre-fetched, saves an API call)
- If `pdf_url` is empty, Strategy 1 is skipped and the chain continues (Unpaywall → EZproxy → Sci-Hub)

In [4]:
# Try to extract EZProxy cookies (optional — for paywalled content)
try:
    COOKIES = extract_cookies_from_browser(browser="firefox")
    if COOKIES:
        logger.info(f"✓ EZProxy cookies loaded ({len(COOKIES)} cookies)")
    else:
        logger.warning("⚠ No EZProxy cookies found — paywalled content may not be accessible")
except Exception as e:
    COOKIES = None
    logger.warning(f"⚠ EZProxy cookies failed: {e}")

results = []
total_records = len(annotated_df)

print(f"Downloading PDFs for {total_records} records (Dryad + Zenodo + SS, no OA filter)...\n")

for idx, row in annotated_df.iterrows():
    doi = row['cited_article_doi']
    record_id = row.get('id', '')
    title = row.get('title', '')
    source = row.get('source', '')
    is_oa = row.get('is_oa')           # True/False/NaN
    pdf_url = row.get('pdf_url') or None  # '' → None so Strategy 1 is skipped cleanly

    record = {
        'article_doi': doi,
        'record_id': record_id,
        'source': source,
        'title': title,
        'is_oa': is_oa,
        'pdf_url_xlsx': pdf_url,
        'downloaded_pdf_path': None,
        'status': 'pending',
        'error': None,
    }

    try:
        pdf_path = download_pdf_with_fallback(
            doi=doi,
            openalex_pdf_url=pdf_url,   # pre-fetched by WU-A3; None → skips Strategy 1
            output_dir=OUTPUT_DIR,
            use_unpaywall=True,
            ezproxy_cookies=COOKIES,
        )

        if pdf_path:
            record['status'] = 'downloaded'
            record['downloaded_pdf_path'] = str(pdf_path.relative_to(OUTPUT_DIR.parent))
            logger.info(f"[{idx+1}/{total_records}] ✓ [{source:18s}] {doi[:40]}...")
        else:
            record['status'] = 'failed'
            record['error'] = 'All download strategies failed'
            logger.warning(f"[{idx+1}/{total_records}] ✗ [{source:18s}] {doi[:40]}... → failed")

    except Exception as e:
        record['status'] = 'error'
        record['error'] = str(e)
        logger.error(f"[{idx+1}/{total_records}] ✗ [{source:18s}] {doi[:40]}... → {e}")

    results.append(record)
    time.sleep(0.3)

results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"Download complete — {len(results_df)} records processed")
print(f"{'='*60}")

INFO:llm_metadata.pdf_download:Strategy 2: Trying Unpaywall for https://doi.org/10.5281/zenodo.5898699


Extracting cookies from Firefox...
✗ No authentication cookies found in browser



INFO:llm_metadata.pdf_download:Strategy 4: Trying Sci-Hub for https://doi.org/10.5281/zenodo.5898699


INFO:Sci-Hub:Failed to fetch pdf with identifier https://doi.org/10.5281/zenodo.5898699 (resolved url None) due to request exception.


ERROR:llm_metadata.pdf_download:All download strategies failed for https://doi.org/10.5281/zenodo.5898699


...

INFO:llm_metadata.pdf_download:Strategy 2: Trying Unpaywall for https://www.semanticscholar.org/paper/9ed892c31685e48936b93f1ad3cdd781e1eb396d


INFO:llm_metadata.pdf_download:Strategy 4: Trying Sci-Hub for https://www.semanticscholar.org/paper/9ed892c31685e48936b93f1ad3cdd781e1eb396d


INFO:Sci-Hub:Failed to fetch pdf with identifier https://www.semanticscholar.org/paper/9ed892c31685e48936b93f1ad3cdd781e1eb396d (resolved url None) due to request exception.


ERROR:llm_metadata.pdf_download:All download strategies failed for https://www.semanticscholar.org/paper/9ed892c31685e48936b93f1ad3cdd781e1eb396d



Download complete — 250 records processed


## Section 4: Manifest & Synthesis by Source

Save the manifest CSV and display download coverage segmented by source (Dryad / Zenodo / Semantic Scholar).

In [5]:
# Save manifest
results_df.to_csv(MANIFEST_PATH, index=False)
print(f"✓ Manifest saved to: {MANIFEST_PATH}")
print(f"✓ PDFs stored in:    {OUTPUT_DIR.resolve()}")

# ── Overall summary ───────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("OVERALL DOWNLOAD SUMMARY")
print(f"{'='*65}")
for status, count in results_df['status'].value_counts().items():
    pct = 100 * count / len(results_df)
    print(f"  {status:20s}: {count:4d}  ({pct:5.1f}%)")

# ── By source ─────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("BY SOURCE")
print(f"{'='*65}")

for src in sorted(results_df['source'].unique()):
    sub = results_df[results_df['source'] == src]
    total = len(sub)
    downloaded = (sub['status'] == 'downloaded').sum()
    failed = sub['status'].isin(['failed', 'error']).sum()
    oa_count = sub['is_oa'].eq(True).sum()
    closed_count = sub['is_oa'].eq(False).sum()
    unknown_count = total - oa_count - closed_count

    print(f"\n  {src.upper()} — {total} records")
    print(f"    Downloaded : {downloaded:3d} / {total}  ({100*downloaded/total:5.1f}%)")
    print(f"    Failed     : {failed:3d} / {total}  ({100*failed/total:5.1f}%)")
    print(f"    OA status  : {oa_count} OA  |  {closed_count} closed  |  {unknown_count} unknown")

    oa_sub = sub[sub['is_oa'] == True]
    closed_sub = sub[sub['is_oa'] == False]
    if len(oa_sub):
        oa_dl = (oa_sub['status'] == 'downloaded').sum()
        print(f"      OA       : {oa_dl}/{len(oa_sub)} downloaded")
    if len(closed_sub):
        cl_dl = (closed_sub['status'] == 'downloaded').sum()
        print(f"      Closed   : {cl_dl}/{len(closed_sub)} downloaded")

# ── Failed detail ─────────────────────────────────────────────────────────────
failed_df = results_df[results_df['status'].isin(['failed', 'error'])]
if len(failed_df):
    print(f"\n{'='*65}")
    print(f"FAILED RECORDS ({len(failed_df)})")
    print(f"{'='*65}")
    for _, row in failed_df.iterrows():
        print(f"  [{row['source']:18s}] {row['article_doi'][:45]}")
        if row['error']:
            print(f"    → {row['error']}")

✓ Manifest saved to: data\pdfs\fuster\manifest.csv
✓ PDFs stored in:    C:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster

OVERALL DOWNLOAD SUMMARY
  downloaded          :  182  ( 72.8%)
  failed              :   68  ( 27.2%)

BY SOURCE

  DRYAD — 37 records
    Downloaded :  36 / 37  ( 97.3%)
    Failed     :   1 / 37  (  2.7%)
    OA status  : 25 OA  |  11 closed  |  1 unknown
      OA       : 25/25 downloaded
      Closed   : 10/11 downloaded

  SEMANTIC_SCHOLAR — 175 records
    Downloaded : 113 / 175  ( 64.6%)
    Failed     :  62 / 175  ( 35.4%)
    OA status  : 46 OA  |  76 closed  |  53 unknown
      OA       : 44/46 downloaded
      Closed   : 69/76 downloaded

  ZENODO — 38 records
    Downloaded :  33 / 38  ( 86.8%)
    Failed     :   5 / 38  ( 13.2%)
    OA status  : 27 OA  |  9 closed  |  2 unknown
      OA       : 25/27 downloaded
      Closed   : 8/9 downloaded

FAILED RECORDS (68)
  [zenodo            ] https://doi.org/10.5281/zenodo.5898699
    → All download strateg